In [0]:
spark.sql("USE CATALOG retail_lakehouse")
spark.sql("USE SCHEMA gold")

In [0]:

from pyspark.sql import functions as F
# read the order_items data from Sliver layer

aggregate_order_df = (
    spark.read.table("retail_lakehouse.sliver.order_items")
    .groupBy("order_id")
    .agg(
        F.round(
            F.sum("line_amount"),
            2
        ).alias("order_total")
    )
)

display(aggregate_order_df)

# unique orders
print("Rows:", aggregate_order_df.count())
aggregate_order_df.groupBy("order_id").count().filter("count > 1").show()




In [0]:

join_df = (
        spark.read.table("retail_lakehouse.sliver.orders")
        .join(
            aggregate_order_df, 
            on="order_id", how="left"
    )
    .join(
        spark.read.table("retail_lakehouse.sliver.payments"),
        on="order_id", how="left"
    )
    .join(
        spark.read.table("retail_lakehouse.sliver.shipping"),
        on="order_id", how="left"
    )
)

display(join_df)

print("rows:", join_df.count())
join_df.groupby("order_id").count().filter("count>1").show()

In [0]:
gold_order_summary_df = (
    join_df
    .select(
        "order_id",
        "customer_id",
        "order_timestamp",
        "order_status",
        "sales_channel",
        "shipping_state",
        "order_total",
        "payment_amount",
        "payment_status",
        "shipment_status",
        "ship_timestamp",
        "delivery_timestamp"
    )
)

display(gold_order_summary_df)

In [0]:
print("Rows:", gold_order_summary_df.count())

gold_order_summary_df \
    .groupBy("order_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

In [0]:
(
    gold_order_summary_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("retail_lakehouse.gold.order_summary")
)